In [18]:
import torch
from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation
import numpy as np
import cv2
import warnings
warnings.filterwarnings("ignore", category=FutureWarning) # huggingface is mad

In [36]:
import importlib
import video_segmentation
importlib.reload(video_segmentation)

<module 'video_segmentation' from 'c:\\Users\\SHANE\\OneDrive\\Documents\\GitHub\\gaitkeeper\\video_segmentation.py'>

In [19]:
import numpy as np
import torch
import cv2
from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation

print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("OpenCV:", cv2.__version__)

NumPy: 1.26.4
Torch: 2.10.0+cpu
OpenCV: 4.8.1


In [20]:
input_video = 'data/test_1.mp4'
output_video = 'data/output_1.mp4'
TARGET_SIZE = 1024
SHIRT_LABEL = 4
BATCH_SIZE = 8

ALPHA = 0.35                  # mask transparency
MASK_COLOR = (0, 255, 0)      # (Blue, Green, Red)

In [21]:
model_name = "mattmdjaga/segformer_b2_clothes"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model
processor = SegformerImageProcessor.from_pretrained("mattmdjaga/segformer_b2_clothes")
model = AutoModelForSemanticSegmentation.from_pretrained("mattmdjaga/segformer_b2_clothes").to(device) # move to gpu if available

model.eval()

print(f'Inference running on {device}')


c:\Users\SHANE\anaconda3\envs\segformer_env\lib\site-packages\transformers\image_processing_base.py:370: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'
  image_processor = cls(**image_processor_dict)
Loading weights: 100%|██████████| 380/380 [00:00<00:00, 640.56it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]            


Inference running on cpu


In [22]:
from video_segmentation import load_segformer_model

# Using load_segformer_model() function to replace the above code cell
processor, model, device = load_segformer_model()
print(f"Inference running on {device}")

Loading weights: 100%|██████████| 380/380 [00:00<00:00, 710.82it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]            


Inference running on cpu


In [23]:
from video_segmentation import get_single_mask_from_video

shirt_tensor = get_single_mask_from_video(input_video, target_resolution=TARGET_SIZE, batch_size=BATCH_SIZE, target_label=SHIRT_LABEL, processor=processor, model=model)

In [24]:
shirt_tensor.shape

torch.Size([469, 1024, 1024])

In [25]:
# Open the original video
cap = cv2.VideoCapture(input_video)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_idx = 0

# Prepare video writer
ret, frame = cap.read()
if not ret:
    raise RuntimeError("Cannot read video")

height, width = frame.shape[:2]
writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)  # reset to first frame

# Iterate over frames
while cap.isOpened() and frame_idx < len(shirt_tensor):
    ret, frame = cap.read()
    if not ret:
        break

    # Resize mask back to original frame size
    mask = shirt_tensor[frame_idx].numpy().astype(np.uint8) * 255
    mask_resized = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)

    # Convert mask to 3 channels
    mask_bgr = np.zeros_like(frame)        # shape (H, W, 3)
    mask_bgr[mask_resized > 0] = MASK_COLOR

    # Overlay mask
    blended = cv2.addWeighted(frame, 1 - ALPHA, mask_bgr, ALPHA, 0)

    writer.write(blended)
    frame_idx += 1

cap.release()
writer.release()
print(f"Output saved to {output_video}")

Output saved to data/output_1.mp4


In [26]:
import video_segmentation
print(video_segmentation.__file__)

c:\Users\SHANE\OneDrive\Documents\GitHub\gaitkeeper\video_segmentation.py


In [37]:
from video_segmentation import write_mask_overlay_video1

write_mask_overlay_video1(
    input_video=input_video,
    output_video="data/output_function1.mp4",
    mask_tensor=shirt_tensor,
    mode="color",
    mask_color=MASK_COLOR,
    alpha=ALPHA
)

In [30]:
overlay_img = cv2.imread("data/noise.jpg")  # BGR

In [31]:
# Code to overlay the pattern onto the original video

cap = cv2.VideoCapture(input_video)
fps = cap.get(cv2.CAP_PROP_FPS)

ret, first_frame = cap.read()
h, w = first_frame.shape[:2]

writer = cv2.VideoWriter(
    output_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h),
)

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# Resize overlay image to video resolution
overlay_img = cv2.resize(overlay_img, (w, h), interpolation=cv2.INTER_LINEAR)

frame_idx = 0

while cap.isOpened() and frame_idx < shirt_tensor.shape[0]:
    ret, frame = cap.read()
    if not ret:
        break

    # Get mask for this frame
    mask = shirt_tensor[frame_idx].numpy().astype(np.uint8)
    mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

    # Expand mask to 3 channels
    mask_3c = mask[:, :, None]  # shape (H, W, 1)

    # Compose overlay only where mask == 1
    composed = frame.copy()
    composed[mask.astype(bool)] = overlay_img[mask.astype(bool)]

    writer.write(composed)
    frame_idx += 1

cap.release()
writer.release()

In [38]:
write_mask_overlay_video1(
    input_video=input_video,
    output_video="data/output_function1.mp4",
    mask_tensor=shirt_tensor,
    mode="pattern",
    overlay_img_path="data/noise.jpg"
)